# Run: resnet18 + MNIST, 20 rounds, checkpointed

One notebook, run top to bottom, single Colab session (free tier).
This is the ONLY notebook you need for this run -- it's a consolidated,
linear version of `00_Setup_and_Crypto_Tests.ipynb` + `CASE1_IID_MNIST.ipynb`
with the smoke test and the 2nd backbone removed, so there's nothing to
skip or get out of order.

**Scope** (see `FIX.md`/`STATUS.md`, 2026-11-15 entry, for the full
reasoning): 1 model (resnet18), MNIST, `n_clients=50`/`sampling_rate=0.25`
unchanged (~13 active clients/round), 20 rounds (down from the full
MVP's 45) -> ~3.8 hours, measured from real per-client timing.

**If Colab disconnects partway through:** re-run from cell 1 again.
Drive stays mounted and the repo stays cloned across a disconnect/
reconnect, and the LAST cell detects the checkpoint already on Drive
and resumes after the last completed round -- it does not restart
from round 0. If you get a `NameError` for a variable partway through
re-running, it just means you haven't reached that cell yet this
pass -- keep running forward, don't skip around.

In [ ]:
# 1. Mount Drive (checkpoint durability + MNIST cache) and clone/pull the repo.
from google.colab import drive
import os
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP

In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).

In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())

In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)

In [ ]:
# 5. Load MNIST, stratified-IID split across clients.
from fl.data_iid import prepare_case1

mnist_cfg = CFG['datasets']['mnist']
case1_data = prepare_case1(seed=CFG['seed'], source=mnist_cfg['source'], csv_path=mnist_cfg.get('csv_path'))
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))

In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")

In [ ]:
# 7. Build the model (resnet18 only, for this run).
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'resnet18'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)
flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)

In [ ]:
# 8. The real run: 20 rounds, checkpointed every round to Drive.
import numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint

REAL_ROUNDS = 20  # down from CFG['mvp_scope']['rounds_per_run'] (45) -- see markdown cell 0

CKPT_DIR = '/content/drive/MyDrive/FL-ZKP/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = f'{CKPT_DIR}/case1_{REAL_MODEL}_r{REAL_ROUNDS}.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"bootstrap-{REAL_MODEL}".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
)
release_gpu_memory()
n_accepted = sum(1 for h in real_history if not h['aborted'])
print(f"done: {len(real_history)} rounds in history, {n_accepted} accepted (not aborted)")